In [1]:
# 01 · MEDIUM 집중 실험 CONFIG
CFG = {
    # GitHub 저장 · 기존 결과 보존, Medium 전용 Release
    'run_name': 'moveboxes_medium_lab_v1',
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes',
    'project_ref': 'main',
    'output_root': '/content/moveboxes_runs',
    'source_run_name': 'moveboxes_stage_act_v1',
    'warm_start': True,

    # T4 / 설치 / 데이터
    'repo_dir': '/content/berlin-marso-hackathon',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],

    # 반복 예산 · 2,000회마다 실제 평가, 최대 20,000회
    'block_iters': 2000,
    'max_blocks': 10,
    'development_episodes': 8,
    'target_accuracy': 0.95,
    'success_streak': 2,
    'plateau_blocks': 4,

    # 작은 모델 · 실행 조건으로 학습, 첫 집기 구간 30% 별도 표집
    'seed': 42,
    'num_demos': None,
    'batch_size': 64,
    'lr': 0.0001,
    'amp': True,
    'history': 16,
    'chunk_size': 16,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'latent_dim': 16,
    'first_pick_fraction': 0.3,

    # 별도 시드와 동일한 200스텝 평가 제한
    'tuning_seed_start': 50000,
    'eval_seed_start': 70000,
    'final_episodes': 100,
    'test_seed_start': 40000,
    'test_episodes': 8,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},

    # 실행 / 출력
    'ensemble_candidates': [1, 4],
    'ensemble_window': 4,
    'temporal_decay': 0.25,
    'gate_threshold': 0.65,
    'stage_threshold': 0.6,
    'console_interval_seconds': 30,
    'team': 'my-team',

}


In [2]:
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
for name in ('build_medium_notebook','medium_lab'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from build_medium_notebook import CONFIG as MEDIUM_DEFAULTS
CFG = dict(MEDIUM_DEFAULTS, **CFG)
from medium_lab import MediumLab, source_bundle
experiment = MediumLab(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


사용 코드: 4494b354b6476e9a35952e3bf2cf26b8a3565b68
코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.


In [3]:
# 03 · GitHub 인증/복원
experiment.connect()
experiment.report()


GitHub 토큰 입력 (이 런타임에서만 사용): ··········
GitHub 복원: 46 files
로컬 작업 경로: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark
실행 노트북 사본: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/moveboxes_medium_colab.ipynb
새 런타임에서는 01~03 셀로 결과를 복원하고, 학습·평가는 04~05 셀 준비 후 실행합니다.
GitHub 백업 완료: common (46 files)
단계 ACT: 학습한 완료/복구 판단 + 집기/운반/놓기 행동, 별도 실험
상태: not_started


{'blocks': [], 'status': 'not_started'}

In [4]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


실행: nvidia-smi
전체 로그: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/sessions/20260914_042634_962090/gpu.log
완료
실행: git clone https://github.com/marso-robotics/berlin-marso-hackathon.git /content/berlin-marso-hackathon
전체 로그: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/sessions/20260914_042634_962090/commands.log
완료
실행: /usr/bin/python3 -m pip install mani-skill==3.0.1 sapien==3.0.3 diffusers==0.38.0 hydra-core omegaconf gymnasium tyro h5py kagglehub tensorboard matplotlib transforms3d imageio[ffmpeg]
전체 로그: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/sessions/20260914_042634_962090/install.log
완료
실행: /usr/bin/python3 -m pip install -e /content/berlin-marso-hackathon
전체 로그: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/sessions/20260914_042634_962090/install_repo.log
완료
설치 완료. 다음 데이터·GPU 확인 셀을 실행하세요.


In [5]:
# 05 · Medium GPU 확인 / 기존 성공 시연과 모델 가져오기
experiment.check_runtime()
experiment.prepare()


실행: /usr/bin/python3 -c import torch
from warehouse_sort.utils import compose_cfg, make_env
assert torch.cuda.is_available(), 'T4 GPU runtime required'
cfg = compose_cfg(['difficulty=medium', 'num_envs=1'])
env, _ = make_env(cfg, 'state', cfg.randomization, num_envs=1, render_mode='rgb_array')
try:
    obs, _ = env.reset(seed=42)
    assert tuple(obs.shape) == (1,72)
    env.step(torch.zeros((1,4),device='cuda'))
    assert env.render() is not None
    print('Medium GPU/state/render OK')
finally:
    env.close()

전체 로그: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/sessions/20260914_042634_962090/medium_gpu_check.log
완료
GitHub 데이터 다운로드: state (65.4 MiB)
easy 200 demos
medium 200 demos
hard 200 demos
GitHub 백업 완료: common (53 files)
GitHub 백업 완료: medium (20 files)
Medium 성공 시연 16개 준비 완료. 반복 학습 셀을 실행하세요.


In [6]:
# 06 · MEDIUM 반복 학습 · 중단 후 같은 셀을 다시 실행하면 이어서 진행
history = experiment.run_blocks()


실행: /usr/bin/python3 stage_train.py /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/stage_train_job.json
전체 로그: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/train_block_01.log
703/20000 loss=0.0981 23.4 it/s ETA=13.7 min
1422/20000 loss=0.0638 23.7 it/s ETA=13.1 min
GitHub 백업 완료: medium (30 files)
완료
실행: /usr/bin/python3 colab_eval_modular.py /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/eval_job.json
전체 로그: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/dev_b01_w1.log
GitHub 백업 완료: medium (34 files)
GitHub 백업 완료: medium (34 files)
GitHub 백업 완료: medium (34 files)
완료
실행: /usr/bin/python3 colab_eval_modular.py /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/eval_job.json
전체 로그: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/dev_b01_w4.log
GitHub 백업 완료: medium (37 files)
GitHub 백업 완료: medium (37 files)
GitHub 백업 완료: medium (37 files)
완료
GitHub 백업 완료: medium (40 files)
Medium 2000회:

In [7]:
# 07 · 최고 모델 8회 별도 테스트 + 영상 + 2회 행동 기록
experiment.test("medium")
experiment.diagnose("medium")


실행: /usr/bin/python3 colab_eval_modular.py /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/eval_job.json
전체 로그: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/test_metrics.log
GitHub 백업 완료: medium (82 files)
GitHub 백업 완료: medium (82 files)
GitHub 백업 완료: medium (82 files)
완료
[medium] 빠른 테스트 8회: 40.6%
결과: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/test_metrics.json / 모델: block_02.pt
빠른 테스트는 최종 가중 점수에 합산하지 않습니다.
실행: /usr/bin/python3 colab_eval_modular.py /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/test_video_job.json
전체 로그: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/test_video.log
완료


[medium] 8회 중 1개 이상 정답 분류 62.5% / 2개 이상 62.5% / 1개만 0.0%
첫 안정적 집기 이후 두 번째 집기 사이클 관측: 100.0%
집기 사이클은 같은 상자를 다시 잡는 경우도 포함합니다. 실제 분류 성과는 위 정답 개수로 확인하세요.
에피소드별 집기 시점·집게 명령 전환: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/test_metrics.json
GitHub 백업 완료: medium (87 files)
실행: /usr/bin/python3 colab_trace.py /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/trace_job.json
전체 로그: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/trace.log
GitHub 백업 완료: medium (90 files)
완료
GitHub 백업 완료: medium (91 files)
[medium] 2회 상태·행동 진단 저장: /content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/traces/de880961bc55152e


In [8]:
# 08 · 학습 곡선과 최고 결과
experiment.report()


 2000회 | 집기 0% | 네 상자 정답 0% | 분류 0.0%
 4000회 | 집기 25% | 네 상자 정답 0% | 분류 15.6%
 6000회 | 집기 12% | 네 상자 정답 0% | 분류 3.1%
 8000회 | 집기 38% | 네 상자 정답 0% | 분류 12.5%
10000회 | 집기 50% | 네 상자 정답 0% | 분류 12.5%
12000회 | 집기 25% | 네 상자 정답 0% | 분류 9.4%
상태: plateau


{'blocks': [{'block': 1,
   'iteration': 2000,
   'best': {'checkpoint': '/content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/checkpoints/block_01.pt',
    'chunk': 1,
    'steps': 1,
    'score': 0.0,
    'policy_config': {'model_config': {'state_dim': 72,
      'history': 16,
      'chunk_size': 16,
      'width': 128,
      'heads': 4,
      'layers': 2,
      'latent_dim': 16},
     'temporal_decay': 0.25,
     'ensemble_window': 1,
     'gate_threshold': 0.65,
     'stage_threshold': 0.6,
     'act_horizon': 1,
     'num_inference_steps': 1},
    'checkpoint_sha256': '5bb74128e845132471610cf81292515ef1137ae22ecd6455a5fde6951776cf87',
    'first_grasp_rate': 0.0,
    'all_sorted_rate': 0.0,
    'block': 1,
    'iteration': 2000},
   'improved': True,
   'trials': [{'checkpoint': '/content/moveboxes_runs/moveboxes_medium_lab_v1_benchmark/medium/checkpoints/block_01.pt',
     'chunk': 1,
     'steps': 1,
     'score': 0.0,
     'policy_config': {'model_config': {'state_d

In [9]:
# 09 · 목표 달성 모델의 별도 시드 최종 평가
experiment.final_evaluation()


네 상자 성공률이 목표에 못 미쳐 최종 평가는 생략합니다. 진단 셀을 확인하세요.


In [10]:
# 10 · 평가 완료한 Medium 모델만 패키징
experiment.package()


패키징할 평가 완료 모델이 없습니다.
